In [5]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch

N = 131072
RADIUS = 0.07
LJ_EPSILON = 0.1
MASS = 200
TEMPERATURE = 1.0
L = 16.0
L_GRID = 32
BX = -8.0
BY = -8.0
BZ = -8.0
N_TEST = 1

LOWER_BOUNDARY = torch.tensor([BX, BY, BZ])
LENGTH = torch.tensor([L, L, L])
GRID_CELL_SIZE = L / L_GRID
# OFFSET = torch.tensor([0.57, 0.57, 0.57])
OFFSET = torch.tensor([0.5, 0.5, 0.5])
NUM_GRIDS = L_GRID ** 3
MAX_GRID_SIZE = (N // NUM_GRIDS) * 4
# GridIndex = wp.vec(MAX_GRID_SIZE, dtype=wp.int32)

DT = 0.001
T_STOP = 10.0

V_MAX = 1e5  # haven't used
D2_MIN = 1e-8

OUTPUT_ROOT = 'torch_mc_simpleGrid_onlyLJ'

positions = torch.rand((N, 3)) * L + LOWER_BOUNDARY
velocities = torch.randn((N, 3)) * np.sqrt(TEMPERATURE / MASS)
forces = torch.zeros((N, 3))

def lj_force(d2: torch.Tensor) -> torch.Tensor:
    inv_d2 = 1.0 / d2
    inv_d6 = inv_d2 * inv_d2 * inv_d2
    inv_d12 = inv_d6 * inv_d6
    f_mag = 24.0 * LJ_EPSILON * (2.0 * inv_d12 - inv_d6) * inv_d2
    return f_mag

t = 0
for step in tqdm(range(int(T_STOP / DT))):
    velocities += 0.5 * forces / MASS * DT    
    positions += velocities * DT
    positions = (positions - LOWER_BOUNDARY) % LENGTH + LOWER_BOUNDARY
    # build grid
    # grid = -torch.ones((NUM_GRIDS, MAX_GRID_SIZE), dtype=torch.int32)
    # grid_sizes = torch.zeros((NUM_GRIDS,), dtype=torch.int32)
    grid_indices = ((positions - LOWER_BOUNDARY) / GRID_CELL_SIZE).to(torch.int32)
    grid_indices = torch.clamp(grid_indices, 0, L_GRID - 1)
    grid_ids = grid_indices[:, 0] * L_GRID * L_GRID + grid_indices[:, 1] * L_GRID + grid_indices[:, 2]
    # for i in range(N):
    #     gid = grid_ids[i].item()
    #     gsize = grid_sizes[gid].item()
    #     if gsize < MAX_GRID_SIZE:
    #         grid[gid, gsize] = i
    #         grid_sizes[gid] += 1


    forces.fill_(0.0)
    # for ib in range(NUM_GRIDS):
    #     ibx = ib // (L_GRID * L_GRID)
    #     iby = ib // L_GRID % L_GRID
    #     ibz = ib % L_GRID
        

100%|██████████| 10000/10000 [00:27<00:00, 359.30it/s]
